# Vibn Fine-tuning on Google Colab

Fine-tunes `Qwen2.5-Coder-7B-Instruct` on your Vibn session transcripts using QLoRA.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU** (free) or **A100** (Colab Pro)
2. Upload your `vibn_training_data.jsonl` using the cell below
3. Run all cells top to bottom
4. Download the output GGUF at the end and load it in Ollama

**Total time:** ~45 min on T4, ~15 min on A100

In [ ]:
# Verify GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout or 'No GPU detected — change runtime type to GPU')

In [ ]:
# Install dependencies (takes ~3 min first time)
!pip install -q unsloth datasets transformers trl peft bitsandbytes
!pip install -q --upgrade unsloth

In [ ]:
# Upload your training data
from google.colab import files
import json, os

print('Upload your vibn_training_data.jsonl file:')
uploaded = files.upload()

data_file = list(uploaded.keys())[0]
examples = [json.loads(l) for l in open(data_file) if l.strip()]
print(f'Loaded {len(examples)} training examples')

# Preview first example
print('\nFirst example preview:')
for turn in examples[0]['conversations'][:3]:
    print(f"  [{turn['from']}]: {turn['value'][:120]}...")

In [ ]:
# Config — adjust if needed
MODEL_NAME = 'Qwen/Qwen2.5-Coder-7B-Instruct'
OUTPUT_DIR = './vibn-lora'
OUTPUT_GGUF = './vibn-coder-q4.gguf'

# LoRA config
LORA_RANK = 16        # Higher = more capacity but more VRAM. 16 is safe for T4.
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Training config
MAX_SEQ_LENGTH = 4096
BATCH_SIZE = 2
GRAD_ACCUM = 4        # Effective batch = BATCH_SIZE * GRAD_ACCUM = 8
EPOCHS = 3
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.05

print(f'Model: {MODEL_NAME}')
print(f'Training examples: {len(examples)}')
print(f'LoRA rank: {LORA_RANK}, Effective batch: {BATCH_SIZE * GRAD_ACCUM}')

In [ ]:
# Load model with Unsloth (4-bit quantized, fits in T4's 16GB)
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # auto-detect: bf16 on A100, fp16 on T4
    load_in_4bit=True,   # QLoRA — halves VRAM usage
)

print(f'Model loaded. Memory: {torch.cuda.memory_allocated()/1e9:.1f} GB used')

In [ ]:
# Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    use_gradient_checkpointing='unsloth',  # saves VRAM
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable/1e6:.1f}M / {total/1e6:.0f}M ({100*trainable/total:.1f}%)')

In [ ]:
# Convert ShareGPT format → ChatML tokens
from datasets import Dataset

def format_conversation(example):
    """Convert ShareGPT conversations to ChatML text."""
    conversations = example['conversations']

    # Map ShareGPT roles to ChatML roles
    role_map = {'system': 'system', 'human': 'user', 'gpt': 'assistant'}

    messages = [
        {'role': role_map.get(c['from'], c['from']), 'content': c['value']}
        for c in conversations
        if c['from'] in role_map
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': text}

dataset = Dataset.from_list(examples)
dataset = dataset.map(format_conversation, remove_columns=['conversations'])

# Split 90/10 train/eval
split = dataset.train_test_split(test_size=0.1, seed=42)
train_ds, eval_ds = split['train'], split['test']

print(f'Train: {len(train_ds)} examples, Eval: {len(eval_ds)} examples')
print('\nSample formatted text (first 400 chars):')
print(train_ds[0]['text'][:400])

In [ ]:
# Train
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_ratio=WARMUP_RATIO,
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        eval_strategy='steps',
        eval_steps=50,
        save_strategy='steps',
        save_steps=100,
        save_total_limit=2,
        load_best_model_at_end=True,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=42,
        report_to='none',
    ),
)

print('Starting training...')
trainer.train()
print('Training complete!')

In [ ]:
# Quick sanity check — run inference on the fine-tuned model
FastLanguageModel.for_inference(model)

test_prompt = [
    {'role': 'system', 'content': 'You are Vibn, an AI coding agent.'},
    {'role': 'user',   'content': 'Read the file src/main.py and summarize what it does.'},
]

inputs = tokenizer.apply_chat_template(
    test_prompt, tokenize=True, add_generation_prompt=True, return_tensors='pt'
).to('cuda')

outputs = model.generate(input_ids=inputs, max_new_tokens=256, use_cache=True)
response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print('Model response:')
print(response)

In [ ]:
# Convert to GGUF (Q4_K_M — good balance of size and quality)
# This is what Ollama uses
print('Saving merged model and converting to GGUF...')
print('This takes ~10 minutes')

model.save_pretrained_gguf(
    'vibn-coder',
    tokenizer,
    quantization_method='q4_k_m',
)

import os
gguf_file = 'vibn-coder/vibn-coder-unsloth.Q4_K_M.gguf'
size_gb = os.path.getsize(gguf_file) / 1e9
print(f'GGUF saved: {gguf_file} ({size_gb:.1f} GB)')

In [ ]:
# Download the GGUF to your machine
from google.colab import files

gguf_file = 'vibn-coder/vibn-coder-unsloth.Q4_K_M.gguf'
print(f'Downloading {gguf_file}...')
print('(This is ~4.5 GB — may take several minutes)')
files.download(gguf_file)

print()
print('Done! Next steps on your Mac:')
print('  1. Move the .gguf file to ~/models/')
print('  2. Run: ollama create vibn-coder -f training/Modelfile')
print('  3. Run: python main.py -m vibn-coder')

## Done!

Back on your Mac:

```bash
# Move the downloaded GGUF
mkdir -p ~/models
mv ~/Downloads/vibn-coder-unsloth.Q4_K_M.gguf ~/models/

# Register with Ollama
ollama create vibn-coder -f /Users/steven/Projects/llm/training/Modelfile

# Run Vibn with your fine-tuned model
cd /Users/steven/Projects/llm
python main.py -m vibn-coder
```